# 03 — Run scenarios

Runs three comparable scenarios on the same agent population and writes per-agent / per-station / per-scenario CSVs plus the required figures.

Scenarios:
1. **S1_real** — real OCM/EIPA charger layout.
2. **S2_clustered** — same number of stations and total ports, clustered tightly near the city centre (radius `SCENARIO.clustered_radius_m`).
3. **S3_distributed** — same number of stations and total ports, on a coarse grid covering the wider study area.

Same RNG seed, same agent count, same horizon → differences in metrics are attributable to layout only.

**Charging-pressure parameters (see `src/config.py → SimulationParams`)** — bumped from the v1 defaults to produce thesis-relevant results:
- horizon: 12 h (was 6 h)
- agents: 500 (was 200)
- trips per agent: 2–4 (was 1–3)
- initial SoC: 0.20–0.55 (was 0.30–0.90)
- low-battery threshold: 0.30 (was 0.20)
- target SoC: 0.70 (was 0.80)
- charger power: 50 kW DC (was 22 kW AC)

In [1]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

from src import config, graph_utils, charger_data, agents as agents_mod, scenarios as scen_mod, simulation as sim_mod, metrics as metrics_mod

print('--- simulation parameters ---')
for k, v in config.SIM.__dict__.items():
    print(f'  {k:30s} = {v}')
print('--- scenario parameters ---')
for k, v in config.SCENARIO.__dict__.items():
    print(f'  {k:30s} = {v}')

--- simulation parameters ---
  seed                           = 42
  dt_minutes                     = 1
  horizon_minutes                = 720
  n_agents                       = 500
  trips_per_agent                = (2, 4)
  battery_capacity_kwh           = 50.0
  consumption_kwh_per_km         = 0.18
  initial_soc_range              = (0.2, 0.55)
  low_battery_threshold          = 0.3
  target_soc                     = 0.7
  charger_power_kw               = 50.0
  morning_peak_min               = 120
  afternoon_peak_min             = 540
  peak_std_min                   = 60
  max_seek_radius_nodes          = 5000
  max_wait_minutes               = 10000
--- scenario parameters ---
  clustered_radius_m             = 400.0
  distributed_grid_size          = 5


In [2]:
G = graph_utils.get_or_build_graph()
df_clean = charger_data.load_clean_chargers()
print(f'graph: {G.number_of_nodes():,} nodes  | clean chargers: {len(df_clean)} | total ports: {int(df_clean["number_of_points"].sum())}')

graph: 1,011 nodes  | clean chargers: 16 | total ports: 27


In [3]:
agents = agents_mod.generate_agents(G)
n_trips = sum(len(a.trips) for a in agents)
print(f'generated {len(agents)} agents, total planned trips = {n_trips}')
import numpy as np
soc_array = np.array([a.soc for a in agents])
print(f'initial SoC: mean={soc_array.mean():.2f}  min={soc_array.min():.2f}  max={soc_array.max():.2f}')

generated 500 agents, total planned trips = 1489
initial SoC: mean=0.37  min=0.20  max=0.55


In [4]:
scenarios = scen_mod.build_all_scenarios(df_clean, G)
for s in scenarios:
    print(f'{s.name}: {len(s.stations)} stations | total ports = {s.stations.total_ports()}')

S1_real: 16 stations | total ports = 27
S2_clustered: 16 stations | total ports = 27
S3_distributed: 16 stations | total ports = 27


In [5]:
import copy
# Each scenario gets a fresh deep copy of the agent list so that state
# from one scenario does not leak into the next.
results = []
for s in scenarios:
    fresh_agents = copy.deepcopy(agents)
    sim = sim_mod.Simulator(G, fresh_agents, s.stations, scenario_name=s.name)
    results.append(sim.run(progress=True))

sim[S3_distributed]: 100%|██████████████████████████████████████████████████████████| 720/720 [00:02<00:00, 359.17it/s]


In [6]:
import pandas as pd
summary_df = pd.DataFrame([r.summary for r in results])
# Reorder for readability.
first_cols = ['scenario', 'n_agents', 'n_stations', 'total_ports',
              'started_charging_events', 'completed_charging_events',
              'total_queued_agents', 'pct_charged_at_least_once',
              'pct_waited_at_least_once']
rest = [c for c in summary_df.columns if c not in first_cols]
summary_df = summary_df[first_cols + rest]
summary_df

,scenario,n_agents,n_stations,total_ports,started_charging_events,completed_charging_events,total_queued_agents,pct_charged_at_least_once,pct_waited_at_least_once,completed_trips,...,mean_waiting_time_min,max_waiting_time_min,median_waiting_time_min,p95_waiting_time_min,mean_waiting_time_among_waiters_min,mean_detour_distance_m,median_detour_distance_m,mean_station_utilisation,total_queue_minutes,max_queue_length
0,S1_real,500,16,27,204,201,139,39.4,27.8,1362,...,38.932,545.0,0.0,277.30,140.043165,538.305533,0.0,0.367101,25537,22
1,S2_clustered,500,16,27,149,146,100,28.6,20.0,1216,...,35.384,610.0,0.0,293.00,176.920000,911.956263,0.0,0.233073,51394,47
2,S3_distributed,500,16,27,212,210,126,40.8,25.2,1396,...,24.996,425.0,0.0,171.55,99.190476,431.806495,0.0,0.317187,15480,20


In [7]:
paths = metrics_mod.write_all_outputs(results)
for k, v in paths.items():
    print(f'{k:36s} → {v}')

scenario_summary                     → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\tables\scenario_summary.csv
agent_results                        → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\tables\agent_results.csv
station_results                      → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\tables\station_results.csv
queue_over_time                      → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\figures\queue_over_time.png
scenario_comparison_waiting_time     → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\figures\scenario_comparison_waiting_time.png
waiting_time_waiters_only            → C:\Users\dsuifh\Downloads\Master Thesis Data SGH\ev-charging-warsaw-simulation-v1\ev_thesis\outputs\figures\waiti